# 03 · TDA and TDDFT from response matrices

这一节只讨论纯泛函的线性响应 TDDFT，并用 PySCF 构造响应矩阵。

目标：

1. 看清 TDA 和 TDDFT 的矩阵结构
2. 手动对角化 PySCF 给出的 `A` / `B` 矩阵
3. 与 PySCF TDA / TDDFT excitation energy 对照

重点不是重新实现 XC kernel，而是理解 TDA 到底丢掉了什么。

---

## 1 · 从 occupied-virtual 激发出发

线性响应 TDDFT 的基底是所有单激发：

$$
i \rightarrow a
$$

其中 $i$ 是 occupied KS orbital，$a$ 是 virtual KS orbital。

如果有 $n_{occ}$ 个占据轨道、$n_{vir}$ 个虚轨道，响应矩阵的维度就是：

$$
n_{ov}=n_{occ}n_{vir}
$$

TDA 和 TDDFT 的差别在矩阵结构。

---

## 2 · TDA vs TDDFT

TDA (Tamm-Dancoff approximation) 只保留激发振幅 $X$：

$$
A X = \omega X
$$

完整 TDDFT / RPA 同时包含激发 $X$ 和退激发 $Y$：

$$
\begin{pmatrix}
A & B \\
-B & -A
\end{pmatrix}
\begin{pmatrix}
X \\
Y
\end{pmatrix}
= \omega
\begin{pmatrix}
X \\
Y
\end{pmatrix}
$$

所以：

| 方法 | 保留什么 | 丢掉什么 | 常见效果 |
|------|----------|----------|----------|
| TDA | $A$ | coupling matrix $B$ | 更稳定，激发能常略高 |
| TDDFT | $A$ 和 $B$ | 无 | 更完整，但可能出现响应不稳定 |

TDA 不是一个新泛函；它是 TDDFT 线性响应方程的近似。

---

## 3 · Ground-state RKS reference

In [ ]:
import numpy as np
from scipy.linalg import eigh, eig
from pyscf import gto, dft, tdscf

HARTREE2EV = 27.211386245988

mol = gto.M(atom="O 0 0 0; H 0 -0.757 0.587; H 0 0.757 0.587", basis="sto-3g")

mf = dft.RKS(mol)
mf.xc = "pbe,pbe"      # pure GGA functional
mf.grids.level = 3
mf.conv_tol = 1e-10
mf.verbose = 0
mf.kernel()

nocc = mol.nelectron // 2
nmo = mf.mo_coeff.shape[1]
nvir = nmo - nocc
nov = nocc * nvir

print(f"RKS/PBE energy = {mf.e_tot:.10f} Hartree")
print(f"nocc = {nocc}, nvir = {nvir}, nov = {nov}")

---

## 4 · PySCF 构造 $A$ 和 $B$

PySCF 的 `get_ab()` 返回四维数组：

```python
A[i,a,j,b], B[i,a,j,b]
```

我们把它 reshape 成二维矩阵后，就可以自己对角化。

In [ ]:
tda_ref = tdscf.TDA(mf)
tda_ref.singlet = True
tda_ref.nstates = 5
tda_ref.verbose = 0

A4, B4 = tda_ref.get_ab()
A = A4.reshape(nov, nov)
B = B4.reshape(nov, nov)

print(f"A4 shape = {A4.shape}  # (i,a,j,b)")
print(f"B4 shape = {B4.shape}  # (i,a,j,b)")
print(f"A shape  = {A.shape}")
print(f"B shape  = {B.shape}")
print(f"max|A-A.T| = {np.max(np.abs(A - A.T)):.2e}")
print(f"max|B-B.T| = {np.max(np.abs(B - B.T)):.2e}")

---

## 5 · 手动 TDA：只对角化 $A$

In [ ]:
tda_energy_manual, tda_vec = eigh(A)
tda_energy_manual = tda_energy_manual[:5]

tda_energy_pyscf, tda_xy = tda_ref.kernel()

print(f"{'state':>5s} {'manual/eV':>12s} {'PySCF/eV':>12s} {'diff/eV':>12s}")
print("-" * 45)
for n, (e1, e2) in enumerate(zip(tda_energy_manual, tda_energy_pyscf), start=1):
    print(f"{n:5d} {e1 * HARTREE2EV:12.6f} {e2 * HARTREE2EV:12.6f} {(e1-e2) * HARTREE2EV:12.2e}")

---

## 6 · 手动 TDDFT：保留 $A$ 和 $B$

完整 TDDFT 的非 Hermitian 响应矩阵为：

$$
M = \begin{pmatrix} A & B \\ -B & -A \end{pmatrix}
$$

它的本征值成对出现：$+\omega$ 和 $-\omega$。物理激发能取正的那一半。

In [ ]:
td_ref = tdscf.TDDFT(mf)
td_ref.singlet = True
td_ref.nstates = 5
td_ref.verbose = 0

td_energy_pyscf, td_xy = td_ref.kernel()

M = np.block([
    [A, B],
    [-B, -A],
])

w, vec = eig(M)
real_w = np.real(w[np.abs(np.imag(w)) < 1e-8])
td_energy_manual = np.sort(real_w[real_w > 1e-8])[:5]

print(f"{'state':>5s} {'manual/eV':>12s} {'PySCF/eV':>12s} {'diff/eV':>12s}")
print("-" * 45)
for n, (e1, e2) in enumerate(zip(td_energy_manual, td_energy_pyscf), start=1):
    print(f"{n:5d} {e1 * HARTREE2EV:12.6f} {e2 * HARTREE2EV:12.6f} {(e1-e2) * HARTREE2EV:12.2e}")

---

## 7 · TDA 和 TDDFT 的数值差别

对同一个 ground-state RKS/PBE 参考态，TDA 和 TDDFT 只在响应方程上不同。

In [ ]:
print(f"{'state':>5s} {'TDA/eV':>12s} {'TDDFT/eV':>12s} {'TDA-TDDFT/eV':>16s}")
print("-" * 55)
for n, (etda, etd) in enumerate(zip(tda_energy_pyscf, td_energy_pyscf), start=1):
    print(f"{n:5d} {etda * HARTREE2EV:12.6f} {etd * HARTREE2EV:12.6f} {(etda-etd) * HARTREE2EV:16.6f}")

---

## 8 · 小结

| 概念 | 含义 |
|------|------|
| single excitation space | occupied-virtual pair $i\to a$ |
| $A$ matrix | 激发-激发 block |
| $B$ matrix | 激发-退激发 coupling block |
| TDA | 解 $AX=\omega X$，忽略 $B$ |
| TDDFT/RPA | 解包含 $A$ 和 $B$ 的响应方程 |
| TDA vs TDDFT | 泛函相同，响应近似不同 |

TDA 常作为更稳定、更便宜的近似；TDDFT 保留退激发耦合，形式上更完整。对于做 WFT 的同学，可以把 TDA 类比成 CIS-like 的线性响应结构，而 TDDFT/RPA 则像加入了 de-excitation coupling 的响应方程。